# 04 - Baseline Models Comparison (Week 4)

Week 4 -- baseline models: Logistic Regression, Random Forest, XGBoost (class_weight='balanced').

In [1]:
import sys; sys.path.insert(0, '../src')
from preprocessing import run_preprocessing_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

data = run_preprocessing_pipeline('../data/raw/creditcard.csv')
X_train, X_test, y_train, y_test = data['X_train'], data['X_test'], data['y_train'], data['y_test']


In [2]:
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, scale_pos_weight=(y_train==0).sum()/max(1,(y_train==1).sum()), random_state=42, eval_metric='aucpr'),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        'model': name,
        'precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'recall': round(recall_score(y_test, pred, zero_division=0), 4),
        'f1': round(f1_score(y_test, pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_test, proba), 4),
        'pr_auc': round(average_precision_score(y_test, proba), 4),
    })
import pandas as pd
comparison = pd.DataFrame(rows)
comparison


,model,precision,recall,f1,roc_auc,pr_auc
0,LogisticRegression,0.0128,0.9268,0.0253,0.9656,0.1072
1,RandomForest,1.0000,0.0244,0.0476,0.9182,0.2946
2,XGBoost,0.8750,0.3415,0.4912,0.8858,0.5267
